# Floriscan Phase A boost — more photos for the weak classes

Rose / tulip / sunflower already work. This notebook **adds** images for lily, carnation, peony, iris, daffodil, hibiscus, cherry blossom into the existing Drive folder, then retrains EfficientNet-B0.

It does **not** re-download TensorFlow flowers or Oxford 102. Keep using `MyDrive/Floriscan/data/species/merged/`.

**Must match Flask:** 384 input, ImageNet mean/std, 10 class names in this order.

After training, copy `floriscan_species.pth` to `D:\\Projects\\Floriscan\\backend\\models\\`.

In [ ]:
# Do not pin opencv 4.8 / numpy 1.26 — that breaks current Colab.
!pip -q install -U timm huggingface_hub
!python -c "import numpy, torch, timm; print('numpy', numpy.__version__); print('torch', torch.__version__); print('timm', timm.__version__)"

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
import os
import random
import shutil
import time

import cv2
import matplotlib.pyplot as plt
import numpy as np
import requests
import timm
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

SPECIES = [
    "rose", "tulip", "lily", "sunflower", "carnation",
    "peony", "iris", "daffodil", "hibiscus", "cherry_blossom",
]
WEAK = ["lily", "carnation", "peony", "iris", "daffodil", "hibiscus", "cherry_blossom"]

DRIVE_ROOT = Path("/content/drive/MyDrive/Floriscan")
MERGED = DRIVE_ROOT / "data" / "species" / "merged"
EXPORT = DRIVE_ROOT / "exports"
CACHE = Path("/content/floriscan_boost_cache")

for name in SPECIES:
    (MERGED / name).mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 384
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def is_image(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTS and path.stat().st_size > 2048


def copy_unique(src: Path, dest_dir: Path, prefix: str) -> bool:
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest = dest_dir / f"{prefix}_{src.name}"
    if dest.exists():
        return False
    shutil.copy2(src, dest)
    return True


def count_class(name: str) -> int:
    folder = MERGED / name
    if not folder.is_dir():
        return 0
    return sum(1 for p in folder.iterdir() if p.is_file() and is_image(p))


def print_counts(title="Images per class"):
    print(title, MERGED)
    for name in SPECIES:
        n = count_class(name)
        mark = "  <-- add more" if name in WEAK and n < 200 else ""
        print(f"  {name:16s} {n}{mark}")


print_counts("Before boost")

## 1. Hugging Face zip — ~1,000 lilies (no Kaggle key)

Downloads the zip and copies only **Lilly / lily**. Skips lotus, orchid, sunflower, tulip.

Do not use `load_dataset()` on this repo — the ImageFolder card is broken.

In [ ]:
import zipfile

from huggingface_hub import hf_hub_download

# Do not use load_dataset() on this repo — its ImageFolder metadata is broken
# (ClassLabel tries to encode the git revision as a class name).
zip_path = Path(
    hf_hub_download(
        repo_id="miladfa7/5-Flower-Types-Classification-Dataset",
        filename="5 Flower Types Classification Dataset-1.zip",
        repo_type="dataset",
    )
)
extract_dir = CACHE / "hf5_flowers"
extract_dir.mkdir(parents=True, exist_ok=True)
marker = extract_dir / ".extracted"
if not marker.exists():
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(extract_dir)
    marker.write_text("ok")

added = 0
skip_parts = ("lotus", "orchid", "tulip", "sunflower")
for path in extract_dir.rglob("*"):
    if not path.is_file() or not is_image(path):
        continue
    blob = str(path).replace("\\", "/").lower()
    if any(part in blob for part in skip_parts):
        continue
    parent = path.parent.name.lower()
    if parent not in {"lilly", "lily"}:
        continue
    added += int(copy_unique(path, MERGED / "lily", "hf5"))
print("HF lilies added", added, "lily total", count_class("lily"))

## 2. Optional Kaggle packs (upload `kaggle.json` if you have one)

- `kausthubkannan/5-flower-types-classification-dataset` — extra Lilly
- `marquis03/flower-classification` — carnation + iris
- `jeffheaton/iris-computer-vision` — more iris
- `sunnysolution1618/flower-hibiscus-rosa-sinensis` — hibiscus close-ups

Skip this cell if you have no Kaggle key. iNaturalist below still fills the gaps.

In [ ]:
# Optional: uncomment the next two lines, run this cell, and choose kaggle.json
# from google.colab import files
# files.upload()

kaggle_json = Path("kaggle.json")
if not kaggle_json.exists() and not Path("/root/.kaggle/kaggle.json").exists():
    print("No kaggle.json — skip Kaggle downloads. iNaturalist still runs next.")
else:
    os.makedirs("/root/.kaggle", exist_ok=True)
    if kaggle_json.exists():
        shutil.copy(kaggle_json, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)
    !pip -q install kaggle

    packs = [
        "kausthubkannan/5-flower-types-classification-dataset",
        "marquis03/flower-classification",
        "jeffheaton/iris-computer-vision",
        "sunnysolution1618/flower-hibiscus-rosa-sinensis",
    ]
    for slug in packs:
        out = CACHE / slug.replace("/", "_")
        out.mkdir(parents=True, exist_ok=True)
        print("\nDownloading", slug)
        !kaggle datasets download -d {slug} -p {out} --unzip -q || echo "failed {slug}"

    hints = {
        "lily": ["lilly", "lily"],
        "carnation": ["carnation", "康乃馨"],
        "iris": ["iris", "鸢尾", "setosa", "versicolor", "virginica"],
        "hibiscus": ["hibiscus", "rosa-sinensis", "rosa_sinensis"],
    }
    skip_parts = {"lotus", "orchid", "rose", "tulip", "sunflower", "daisy", "dandelion", "water"}

    copied = {name: 0 for name in hints}
    for path in CACHE.rglob("*"):
        if not path.is_file() or not is_image(path):
            continue
        blob = str(path).lower()
        if any(part in blob for part in skip_parts):
            continue
        for dest_name, keys in hints.items():
            if any(key.lower() in blob for key in keys):
                copied[dest_name] += int(copy_unique(path, MERGED / dest_name, "kag"))
                break
    print("Kaggle copied", copied)

## 3. iNaturalist research-grade photos

Fills peony, cherry blossom, daffodil, hibiscus, carnation, iris, lily. About **400** close-ups per taxon. Classes that already have 500+ images (lily after Hugging Face) are skipped. Be polite to the API (1s sleep between pages).

In [ ]:
INAT = {
    "lily": "Lilium",
    "carnation": "Dianthus caryophyllus",
    "peony": "Paeonia lactiflora",
    "iris": "Iris germanica",
    "daffodil": "Narcissus",
    "hibiscus": "Hibiscus rosa-sinensis",
    "cherry_blossom": "Prunus serrulata",
}
INAT_TARGET = 400
INAT_HEADERS = {"User-Agent": "FloriscanStudentProject/1.0 (education; Colab)"}


def download_inat(dest_name: str, taxon: str, target: int) -> int:
    dest = MERGED / dest_name
    dest.mkdir(parents=True, exist_ok=True)
    already = count_class(dest_name)
    if already >= 500:
        print(dest_name, "already has", already, "— skip iNat")
        return 0
    added = 0
    page = 1
    while added < target:
        try:
            resp = requests.get(
                "https://api.inaturalist.org/v1/observations",
                params={
                    "taxon_name": taxon,
                    "photos": "true",
                    "quality_grade": "research",
                    "per_page": 50,
                    "page": page,
                    "order_by": "votes",
                },
                headers=INAT_HEADERS,
                timeout=60,
            )
            resp.raise_for_status()
        except Exception as exc:
            print("iNat request failed", dest_name, exc)
            break
        results = resp.json().get("results") or []
        if not results:
            break
        for obs in results:
            for photo in obs.get("photos") or []:
                url = (photo.get("url") or "").replace("square", "medium")
                if not url:
                    continue
                out = dest / f"inat_{obs['id']}_{photo['id']}.jpg"
                if out.exists():
                    continue
                try:
                    img = requests.get(url, headers=INAT_HEADERS, timeout=30)
                    if img.status_code == 200 and len(img.content) > 2048:
                        out.write_bytes(img.content)
                        added += 1
                except Exception:
                    continue
                if added >= target:
                    break
            if added >= target:
                break
        page += 1
        time.sleep(1.0)
        if page > 25:
            break
    print(f"iNat {dest_name:16s} +{added}  total {count_class(dest_name)}")
    return added


for dest_name, taxon in INAT.items():
    download_inat(dest_name, taxon, INAT_TARGET)
print_counts("After iNaturalist")

## 4. Train (continues from your existing `.pth` if present)

Uses a **weighted sampler** so lily / peony / cherry are not drowned by rose / tulip / sunflower.

In [ ]:
print_counts("Ready to train")
thin = [name for name in SPECIES if count_class(name) < 80]
if thin:
    print("WARNING still thin (<80). Train anyway, but expect weak recall:", thin)
else:
    print("All classes have at least 80 images.")

class FlowerFolder(Dataset):
    def __init__(self, items, train=False):
        self.items = items
        self.train = train

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        path, label = self.items[index]
        img = cv2.imread(str(path))
        if img is None:
            img = np.zeros((IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (IMAGE_SIZE, IMAGE_SIZE))
        if self.train:
            if random.random() < 0.5:
                img = cv2.flip(img, 1)
            if random.random() < 0.4:
                angle = random.uniform(-18, 18)
                matrix = cv2.getRotationMatrix2D((IMAGE_SIZE / 2, IMAGE_SIZE / 2), angle, 1.0)
                img = cv2.warpAffine(img, matrix, (IMAGE_SIZE, IMAGE_SIZE), borderMode=cv2.BORDER_REFLECT)
            hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.float32)
            hsv[:, :, 2] *= random.uniform(0.85, 1.15)
            hsv[:, :, 2] = np.clip(hsv[:, :, 2], 0, 255)
            img = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        img = img.astype(np.float32) / 255.0
        img = (img - MEAN) / STD
        tensor = torch.from_numpy(img).permute(2, 0, 1)
        return tensor, label


train_items, val_items, test_items = [], [], []
print("per-class 70/15/15 split:")
for class_id, name in enumerate(SPECIES):
    files = [p for p in (MERGED / name).iterdir() if p.is_file() and is_image(p)]
    files.sort()
    random.shuffle(files)
    n = len(files)
    n_train = int(0.70 * n)
    n_val = int(0.15 * n)
    train_files = files[:n_train]
    val_files = files[n_train:n_train + n_val]
    test_files = files[n_train + n_val:]
    train_items.extend((path, class_id) for path in train_files)
    val_items.extend((path, class_id) for path in val_files)
    test_items.extend((path, class_id) for path in test_files)
    print(f"  {name:16s} n={n:4d}  train {len(train_files):4d}  val {len(val_files):4d}  test {len(test_files):4d}")

random.shuffle(train_items)
print("totals", len(train_items), len(val_items), len(test_items))

counts = np.zeros(len(SPECIES), dtype=np.float64)
for _, label in train_items:
    counts[label] += 1
counts = np.maximum(counts, 1.0)
sample_w = [1.0 / counts[label] for _, label in train_items]
sampler = WeightedRandomSampler(sample_w, num_samples=len(train_items), replacement=True)

train_loader = DataLoader(FlowerFolder(train_items, train=True), batch_size=16, sampler=sampler, num_workers=2)
val_loader = DataLoader(FlowerFolder(val_items, train=False), batch_size=16, shuffle=False, num_workers=2)
test_loader = DataLoader(FlowerFolder(test_items, train=False), batch_size=16, shuffle=False, num_workers=2)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device", device)

model = timm.create_model("efficientnet_b0", pretrained=True, num_classes=len(SPECIES))
prev = EXPORT / "floriscan_species.pth"
if prev.is_file():
    try:
        model.load_state_dict(torch.load(prev, map_location="cpu"))
        print("Loaded previous weights", prev)
    except Exception as exc:
        print("Could not load previous weights, training from ImageNet:", exc)
model.to(device)

class_w = torch.tensor((counts.max() / counts).astype(np.float32), device=device)
print("class weights", {SPECIES[i]: round(float(class_w[i]), 2) for i in range(len(SPECIES))})
criterion = nn.CrossEntropyLoss(weight=class_w)

for name, param in model.named_parameters():
    param.requires_grad = name.startswith("classifier")
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)


def run_epoch(loader, train=True):
    model.train(train)
    total_loss = 0.0
    correct = 0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        if train:
            optimizer.zero_grad()
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss = criterion(logits, labels)
            if train:
                loss.backward()
                optimizer.step()
        total_loss += loss.item() * labels.size(0)
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / max(total, 1), correct / max(total, 1)


best_val = 0.0
best_path = EXPORT / "floriscan_species.pth"
for epoch in range(6):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"head {epoch+1:02d}  train {train_acc:.3f}  val {val_acc:.3f}  val_loss {val_loss:.4f}")
    if val_acc >= best_val:
        best_val = val_acc
        torch.save(model.state_dict(), best_path)

for param in model.parameters():
    param.requires_grad = True
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for epoch in range(8):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"ft   {epoch+1:02d}  train {train_acc:.3f}  val {val_acc:.3f}  val_loss {val_loss:.4f}")
    if val_acc >= best_val:
        best_val = val_acc
        torch.save(model.state_dict(), best_path)

print("best val", best_val, "saved", best_path)

In [ ]:
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()
y_true, y_pred = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        y_true.extend(labels.tolist())
        y_pred.extend(logits.argmax(1).cpu().tolist())

label_ids = list(range(len(SPECIES)))
print("test images per class:")
for class_id, name in enumerate(SPECIES):
    print(f"  {name:16s} {y_true.count(class_id)}")

print(
    classification_report(
        y_true,
        y_pred,
        labels=label_ids,
        target_names=SPECIES,
        digits=3,
        zero_division=0,
    )
)
matrix = confusion_matrix(y_true, y_pred, labels=label_ids)
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(matrix, cmap="Greens")
ax.set_xticks(range(len(SPECIES)), SPECIES, rotation=45, ha="right")
ax.set_yticks(range(len(SPECIES)), SPECIES)
ax.set_title("Floriscan species confusion matrix (boost)")
fig.colorbar(im)
plt.tight_layout()
fig.savefig(EXPORT / "species_confusion_matrix_boost.png", dpi=140)
plt.show()
print("Copy this file to D:\\Projects\\Floriscan\\backend\\models\\floriscan_species.pth")
print(best_path)